In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from lazypredict.Supervised import LazyClassifier

In [4]:
TARGET_ROWS = 30000
TOLERANCE = 0.15  # skip any single repo that alone would push total too far past target

df = pd.read_csv("../data/processed/pr_snapshots_clean.csv")

repo_sizes = df.groupby("repo_key").size().sample(frac=1, random_state=42)
keep_repos, total = [], 0
for repo, size in repo_sizes.items():
    if total >= TARGET_ROWS:
        break
    if total + size > TARGET_ROWS * (1 + TOLERANCE):
        continue
    keep_repos.append(repo)
    total += size

df = df[df.repo_key.isin(keep_repos)].copy()
print(f"{len(keep_repos)} repos, {len(df)} rows")

y = df.pop("merged_before_next")
X = df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

5 repos, 30867 rows
shape: (30867, 17), target distribution: {0: 19349, 1: 11518}


In [5]:
# group split: whole repos held out, never split across train/test -- real generalization check
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["repo_key"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"train repos: {df.repo_key.iloc[train_idx].nunique()}, test repos: {df.repo_key.iloc[test_idx].nunique()}")

clf = LazyClassifier(verbose=0, ignore_warnings=True, predictions=False)
models, preds = clf.fit(X_train, X_test, y_train, y_test)
models

train repos: 4, test repos: 1


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
ExtraTreesClassifier,0.626412,0.551403,0.584997,0.603792,0.599297,0.626412,4.666771
NuSVC,0.571781,0.544494,0.545926,0.576759,0.583734,0.571781,225.813304
XGBClassifier,0.624153,0.544456,0.576329,0.597578,0.593455,0.624153,0.718725
BaggingClassifier,0.606695,0.537662,0.555364,0.589206,0.582494,0.606695,2.030129
SVC,0.643869,0.537377,0.548717,0.587731,0.602890,0.643869,44.643417
LGBMClassifier,0.628260,0.535880,0.569162,0.589135,0.589222,0.628260,0.382051
RandomForestClassifier,0.605052,0.532925,0.569246,0.585225,0.578240,0.605052,5.902475
KNeighborsClassifier,0.574040,0.532757,0.547141,0.573926,0.573812,0.574040,4.112991
AdaBoostClassifier,0.590470,0.527148,0.571922,0.577567,0.570690,0.590470,1.386552


In [6]:
# Sum up the total time taken by all models in seconds
total_seconds = models['Time Taken'].sum()

# Convert it to minutes and seconds for better readability
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print(f"Total LazyPredict execution time: {total_seconds:.2f} seconds")
print(f"Formatted time: {minutes} minutes and {seconds} seconds")

Total LazyPredict execution time: 384.21 seconds
Formatted time: 6 minutes and 24 seconds
